# Notebook 2 — Engenharia de Features: da Curva DeltaQ(V) ao Preditor

Neste notebook construimos as **features** que o artigo usa para prever a vida util.
O ponto de partida e a curva Delta Q(V) do Notebook 1.

**Resultado central do artigo:** a variancia de Delta Q_{100-10}(V), em escala
logaritmica, tem correlacao de **rho = -0,93** com o logaritmo da vida util —
calculada a partir dos primeiros 100 ciclos, antes de qualquer queda de capacidade.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from scipy.stats import pearsonr

plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (11, 4),
                     'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

def simular_curva_bruta(ciclo, taxa_deg=0.001, semente=42):
    rng = np.random.default_rng(semente + ciclo * 7)
    n_pts = 190 + rng.integers(-25, 40)
    deg = taxa_deg * ciclo
    V = np.linspace(3.5, 2.0, n_pts) + rng.normal(0, 0.003, n_pts)
    V = np.clip(V, 2.0, 3.5)
    V = np.sort(V)[::-1]
    x = (3.5 - V) / 1.5
    p1 = 1 / (1 + np.exp(-28 * (x - 0.12 - deg * 0.25)))
    p2 = 1 / (1 + np.exp(-18 * (x - 0.62 - deg * 0.40)))
    p3 = 1 / (1 + np.exp(-12 * (x - 0.92)))
    Q = 1.1 * (1 - 0.08 * deg) * (0.42 * p1 + 0.46 * p2 + 0.12 * p3)
    Q += rng.normal(0, 0.0015, n_pts)
    return V, np.clip(Q, 0, None)

def padronizar_curva(V_bruto, Q_bruto, n_pontos=1000):
    V_grid = np.linspace(3.5, 2.0, n_pontos)
    V_ord = V_bruto[::-1].copy(); Q_ord = Q_bruto[::-1].copy()
    _, idx = np.unique(V_ord, return_index=True)
    return V_grid, UnivariateSpline(V_ord[idx], Q_ord[idx], s=0, ext='const')(V_grid)

def get_Q(ciclo, taxa, semente):
    return padronizar_curva(*simular_curva_bruta(ciclo, taxa, semente))

print('Funcoes carregadas.')


## 1. A Transformacao Delta Q(V)

A curva diferencial e definida como:

    Delta Q_{i-j}(V) = Q_i(V) - Q_j(V)

Com ambos os vetores na mesma grade de 1.000 pontos, a subtracao e elemento a elemento.
O artigo usa principalmente:

    Delta Q_{100-10}(V) = Q_{ciclo 100}(V) - Q_{ciclo 10}(V)

**Interpretacao fisica:**
- Bateria saudavel: Delta Q(V) ~ 0 em toda a faixa de tensao
- Bateria degradando silenciosamente: a **distribuicao** de onde a capacidade
  e entregue muda, mesmo que a capacidade total seja quase igual
- O mecanismo: **LAMdeNE** desloca os plateaus de grafite de forma nao uniforme


In [ ]:
taxa = 0.002
V_grid, Q10  = get_Q(10,  taxa, 42)
_,      Q100 = get_Q(100, taxa, 42)
dQ = Q100 - Q10

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(V_grid, Q10,  lw=2, color='steelblue', label='Q ciclo 10')
ax.plot(V_grid, Q100, lw=2, color='tomato', linestyle='--', label='Q ciclo 100')
ax.set_xlabel('Tensao (V)'); ax.set_ylabel('Q(V) (Ah)')
ax.set_title('Curvas Q(V) para ciclos 10 e 100'); ax.legend()

ax = axes[1]
ax.plot(V_grid, dQ, lw=2, color='purple')
ax.axhline(0, color='black', lw=0.7, linestyle=':')
ax.fill_between(V_grid, dQ, 0, where=(dQ > 0), alpha=0.2, color='green', label='Ganho')
ax.fill_between(V_grid, dQ, 0, where=(dQ < 0), alpha=0.2, color='red',   label='Perda')
ax.set_xlabel('Tensao (V)'); ax.set_ylabel('Delta Q_{100-10}(V) (Ah)')
ax.set_title('Curva diferencial Delta Q(V)'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Variancia: {np.var(dQ):.2e} Ah^2  |  Minimo: {dQ.min():.5f} Ah  |  Media: {dQ.mean():.5f} Ah')


## 2. Vida Longa vs. Vida Curta: o Sinal e Visivel desde o Ciclo 100

Celulas com vida util curta ja mostram, nos primeiros 100 ciclos, uma curva
Delta Q(V) mais "agitada" (maior variancia) do que celulas longevas.

A degradacao intensa (LAMdeNE forte) desloca os plateaus de forma mais pronunciada,
gerando uma curva diferencial que oscila mais — mesmo sem queda visivel de capacidade.


In [ ]:
taxa_longa = 0.0005   # vida ~2000 ciclos
taxa_curta = 0.0050   # vida ~400 ciclos

V_grid, dQ_longa = (lambda: (lambda V,Qi,Qj: (V, Qi-Qj))(*get_Q(100,taxa_longa,10), get_Q(10,taxa_longa,10)[1]))()
V_g2, Q100c = get_Q(100, taxa_curta, 20); _, Q10c = get_Q(10, taxa_curta, 20); dQ_curta = Q100c - Q10c

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(V_grid, dQ_longa, lw=2, color='steelblue', label=f'Vida longa  | Var={np.var(dQ_longa):.2e}')
ax.plot(V_grid, dQ_curta, lw=2, color='tomato', linestyle='--', label=f'Vida curta  | Var={np.var(dQ_curta):.2e}')
ax.axhline(0, color='black', lw=0.7, linestyle=':')
ax.set_xlabel('Tensao (V)'); ax.set_ylabel('Delta Q_{100-10}(V) (Ah)')
ax.set_title('Comparacao de Delta Q(V)'); ax.legend(fontsize=9)

ax = axes[1]
ax.hist(dQ_longa, bins=50, color='steelblue', alpha=0.6, label='Vida longa')
ax.hist(dQ_curta, bins=50, color='tomato',    alpha=0.6, label='Vida curta')
ax.set_xlabel('Valor de Delta Q(V) (Ah)'); ax.set_ylabel('Frequencia')
ax.set_title('Distribuicao dos valores — maior espalhamento = maior variancia')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

razao = np.var(dQ_curta) / np.var(dQ_longa)
print(f'Variancia (curta) / Variancia (longa) = {razao:.1f}x')


## 3. Estatisticas de Resumo: de 1.000 Numeros para 1

A curva Delta Q(V) tem 1.000 pontos. O artigo extrai **escalares** que resumem
o comportamento global para uso no modelo linear:

| Feature | Definicao | Interpretacao |
|---------|-----------|---------------|
| `log10(Var[DeltaQ])` | Logaritmo da variancia | Quanto a curva oscila — **preditor dominante** |
| `min(DeltaQ)` | Valor minimo da curva | Regiao de tensao mais afetada pela degradacao |
| `mean(DeltaQ)` | Media da curva | Mudanca media de capacidade entre os ciclos |

A transformacao logaritmica e necessaria porque a relacao entre variancia
e vida util e **log-linear** (nao linear direta).


In [ ]:
def extrair_features(dQ):
    """Extrai as tres features escalares de uma curva Delta Q(V)."""
    var_dQ = np.var(dQ)
    return {
        'log10(Var[DeltaQ])':  np.log10(var_dQ) if var_dQ > 0 else -np.inf,
        'min(DeltaQ)':         np.min(dQ),
        'mean(DeltaQ)':        np.mean(dQ),
    }

# Demonstracao para a celula de taxa=0.002
feats = extrair_features(dQ)
for nome, valor in feats.items():
    print(f'  {nome:28s}: {valor:+.6f}')

print()
print('Esses 3 numeros capturam a assinatura de degradacao precoce da celula.')


## 4. O Resultado-Chave: Correlacao com a Vida Util

Simulamos **80 celulas** com diferentes taxas de degradacao e calculamos
a correlacao entre `log(Var[DeltaQ])` e o logaritmo da vida util.

No artigo (dados reais de 124 celulas):
- Correlacao da capacidade no ciclo 2:   **rho = -0,06** (estatisticamente inutil)
- Correlacao de `log(Var[DeltaQ])`:      **rho = -0,93** resultado central


In [ ]:
np.random.seed(0)
n = 80
taxas = np.random.uniform(0.0003, 0.006, n)
vidas = np.clip((0.080 / taxas) * np.exp(np.random.normal(0, 0.12, n)), 100, 3000).astype(int)

log_var = []
for i, (taxa, seed) in enumerate(zip(taxas, range(n))):
    _, Q10i  = get_Q(10,  taxa, seed * 13 + 1)
    _, Q100i = get_Q(100, taxa, seed * 13 + 1)
    log_var.append(np.log10(np.var(Q100i - Q10i)))

log_var  = np.array(log_var)
log_vida = np.log10(vidas)
rho, pval = pearsonr(log_var, log_vida)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sc = ax.scatter(log_var, log_vida, c=vidas, cmap='RdYlGn', s=55,
                edgecolors='gray', linewidths=0.4)
plt.colorbar(sc, ax=ax, label='Vida util (ciclos)')
m, b = np.polyfit(log_var, log_vida, 1)
x_fit = np.linspace(log_var.min(), log_var.max(), 100)
ax.plot(x_fit, m * x_fit + b, 'k--', lw=1.5, label=f'Tendencia (rho={rho:.2f})')
ax.set_xlabel('log10(Var[DeltaQ_{100-10}(V)])')
ax.set_ylabel('log10(Vida util em ciclos)')
ax.set_title(f'Correlacao log-linear\nrho = {rho:.2f}  (p = {pval:.2e})')
ax.legend()

ax = axes[1]
ax.scatter(log_var, vidas, c=vidas, cmap='RdYlGn', s=55, edgecolors='gray', lw=0.4)
ax.set_xlabel('log10(Var[DeltaQ])'); ax.set_ylabel('Vida util (ciclos) escala linear')
ax.set_title('Mesmo dado em escala linear')

plt.tight_layout(); plt.show()

print(f'Correlacao de Pearson rho = {rho:.3f} (artigo real: -0.93)')
print('Celulas com MAIOR variancia -> degradam MAIS RAPIDO -> vida util MENOR')


## 5. Resumo

O preditor principal em 3 linhas:

```python
V_grid, Q_100 = padronizar_curva(*medir_descarga(ciclo=100))
V_grid, Q_010 = padronizar_curva(*medir_descarga(ciclo=10))
feature = np.log10(np.var(Q_100 - Q_010))    # correlacao -0.93 com log(vida util)
```

Proximo notebook: como usar essa feature num modelo Elastic Net
para prever quantitativamente a vida util.
